In [ ]:
!pip install transformers torch accelerate bitsandbytes

In [ ]:
import sqlite3

# Create a database
conn = sqlite3.connect("travel.db")
cursor = conn.cursor()

# Create hotels table
cursor.execute("""
CREATE TABLE hotels (
    city TEXT,
    hotel_name TEXT,
    price_per_night INTEGER
)
""")

# Create attractions table
cursor.execute("""
CREATE TABLE attractions (
    city TEXT,
    attraction_name TEXT,
    ticket_price INTEGER
)
""")

conn.commit()

print("Database created successfully!")


In [ ]:
# Insert hotel data
hotels_data = [
    ("Paris", "Hotel Lumiere", 220),
    ("Paris", "Budget Paris Inn", 120),
    ("Rome", "Roma Palace", 180),
    ("Rome", "Colosseum Stay", 130),
    ("Tokyo", "Shibuya Hotel", 200),
    ("Tokyo", "Tokyo Budget Lodge", 110)
]

cursor.executemany("INSERT INTO hotels VALUES (?, ?, ?)", hotels_data)

# Insert attraction data
attractions_data = [
    ("Paris", "Eiffel Tower", 30),
    ("Paris", "Louvre Museum", 25),
    ("Rome", "Colosseum", 20),
    ("Rome", "Vatican Museum", 22),
    ("Tokyo", "Tokyo Tower", 15),
    ("Tokyo", "Shinjuku Gyoen", 10)
]

cursor.executemany("INSERT INTO attractions VALUES (?, ?, ?)", attractions_data)

conn.commit()

print("Sample travel data inserted!")


In [ ]:
def search_hotels(city, max_price=None):
    if max_price:
        cursor.execute(
            "SELECT * FROM hotels WHERE city=? AND price_per_night<=?",
            (city, max_price)
        )
    else:
        cursor.execute(
            "SELECT * FROM hotels WHERE city=?",
            (city,)
        )

    return cursor.fetchall()


In [ ]:
print(search_hotels("Paris"))

In [ ]:
def search_attractions(city):
    cursor.execute(
        "SELECT * FROM attractions WHERE city=?",
        (city,)
    )

    return cursor.fetchall()


In [ ]:
print(search_attractions("Rome"))


In [ ]:
def simple_assistant(query):
    query = query.lower()

    if "hotel" in query:
        if "paris" in query:
            return search_hotels("Paris")
        if "rome" in query:
            return search_hotels("Rome")
        if "tokyo" in query:
            return search_hotels("Tokyo")

    if "attraction" in query:
        if "paris" in query:
            return search_attractions("Paris")
        if "rome" in query:
            return search_attractions("Rome")
        if "tokyo" in query:
            return search_attractions("Tokyo")

    return "Sorry, I couldn't understand."


In [ ]:
print(simple_assistant("Show me hotels in Rome"))


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "microsoft/phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")


In [ ]:
def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
print(generate_response("Suggest a 3-day trip to Rome."))


In [ ]:
def travel_assistant(query):
    query_lower = query.lower()

    # Tool usage
    if "hotel" in query_lower:
        if "paris" in query_lower:
            results = search_hotels("Paris")
        elif "rome" in query_lower:
            results = search_hotels("Rome")
        elif "tokyo" in query_lower:
            results = search_hotels("Tokyo")
        else:
            results = []

        prompt = f"""
        The user asked: {query}

        Here are the available hotels from the database:
        {results}

        Provide a helpful recommendation using this data.
        """

        return generate_response(prompt)

    # If no tool needed
    return generate_response(query)


In [ ]:
print(travel_assistant("Show me hotels in Rome under $200"))


In [ ]:
!git clone https://github.com/lavanya-4/Travel_Trip_Advisor.git

In [ ]:
%cd Travel_Trip_Advisor

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content

In [ ]:
%cd /content/Travel_Trip_Advisor

In [ ]:
!ls "/content/drive/My Drive/Colab Notebooks"